In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Lab 11: CNN Model for Time-Series Forecasting\n",
    "\n",
    "**Student Name:** M.Sufyan Khan  \n",
    "**Registration Number:** 22JZELE0477  \n",
    "**Course:** Machine Learning Lab  \n",
    "**Supervisor:** Engr. Irshad Ullah  \n",
    "**University:** UET Nowshera Campus  \n",
    "\n",
    "\n",
    "## Learning Objectives\n",
    "- Import TensorFlow/Keras, Scikit-learn metrics, and custom time-series utilities.\n",
    "- Define a CNN architecture using Conv1D layers for sequential data.\n",
    "- Configure callbacks, optimizer, checkpoints, and training paths.\n",
    "- Load train, validation, and test datasets for forecasting.\n",
    "- Evaluate CNN forecasting performance using standard regression metrics.\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Section 1: Working Directory and Library Imports\n",
    "This section sets the Lab 10/11 path and imports all libraries required for CNN-based time-series forecasting.\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 2,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "os.chdir(r'D:\\Machine Learning\\ML Lab\\Lab 11')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 3,
   "metadata": {},
   "outputs": [],
   "source": [
    "from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score\n",
    "from timeseires.utils.to_split import to_split\n",
    "from timeseires.utils.multivariate_multi_step import multivariate_multi_step\n",
    "from timeseires.utils.multivariate_single_step import multivariate_single_step\n",
    "from timeseires.utils.univariate_multi_step import univariate_multi_step\n",
    "from timeseires.utils.univariate_single_step import univariate_single_step\n",
    "from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS\n",
    "from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint\n",
    "from tensorflow.keras.callbacks import ModelCheckpoint\n",
    "from timeseires.callbacks.TrainingMonitor import TrainingMonitor\n",
    "from tensorflow.keras.optimizers import Adam\n",
    "from tensorflow.keras.optimizers import SGD\n",
    "from tensorflow.keras.models import load_model\n",
    "from tensorflow.keras.layers import LSTM, Bidirectional, Add\n",
    "from tensorflow.keras.layers import BatchNormalization\n",
    "from tensorflow.keras.layers import Conv1D,TimeDistributed\n",
    "from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPooling1D,Concatenate,AveragePooling1D, GlobalMaxPooling1D, Input\n",
    "from tensorflow.keras.models import Sequential,Model\n",
    "import pandas as pd\n",
    "import time, pickle\n",
    "import numpy as np\n",
    "import tensorflow.keras.backend as K\n",
    "import tensorflow\n",
    "from tensorflow.keras.layers import Input, Reshape, Lambda\n",
    "from tensorflow.keras.layers import Layer, Flatten, LeakyReLU, concatenate, Dense\n",
    "from tensorflow.keras.regularizers import l2\n",
    "import glob\n",
    "import h5py\n",
    "import matplotlib.pyplot as plt\n",
    "from keras.callbacks import Callback"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 4,
   "metadata": {},
   "outputs": [],
   "source": [
    "#lookback = 24\n",
    "model = None\n",
    "start_epoch = 0\n",
    "time_steps=24\n",
    "num_features=21"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 5,
   "metadata": {},
   "outputs": [],
   "source": [
    "def CNN():\n",
    "    input_data = Input(shape=(time_steps, num_features))\n",
    "    x1 = Conv1D(16, 2, activation=\"relu\")(input_data)\n",
    "    x2 = Conv1D(16, 2, activation=\"relu\")(x1)\n",
    "    flatten = Flatten()(x2)\n",
    "    output_data = Dense(1)(flatten)\n",
    "    model = Model(input_data, output_data)\n",
    "    return model"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 6,
   "metadata": {},
   "outputs": [
    {
     "data": {
      "text/html": [
       "<pre style=\"white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace\"><span style=\"font-weight: bold\">Model: \"functional\"</span>\n",
       "</pre>\n"
      ],
      "text/plain": [
       "\u001b[1mModel: \"functional\"\u001b[0m\n"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    },
    {
     "data": {
      "text/html": [
       "<pre style=\"white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace\">┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓\n",
       "┃<span style=\"font-weight: bold\"> Layer (type)                    </span>┃<span style=\"font-weight: bold\"> Output Shape           </span>┃<span style=\"font-weight: bold\">       Param # </span>┃\n",
       "┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩\n",
       "│ input_layer (<span style=\"color: #0087ff; text-decoration-color: #0087ff\">InputLayer</span>)        │ (<span style=\"color: #00d7ff; text-decoration-color: #00d7ff\">None</span>, <span style=\"color: #00af00; text-decoration-color: #00af00\">24</span>, <span style=\"color: #00af00; text-decoration-color: #00af00\">21</span>)         │             <span style=\"color: #00af00; text-decoration-color: #00af00\">0</span> │\n",
       "├─────────────────────────────────┼────────────────────────┼───────────────┤\n",
       "│ conv1d (<span style=\"color: #0087ff; text-decoration-color: #0087ff\">Conv1D</span>)                 │ (<span style=\"color: #00d7ff; text-decoration-color: #00d7ff\">None</span>, <span style=\"color: #00af00; text-decoration-color: #00af00\">23</span>, <span style=\"color: #00af00; text-decoration-color: #00af00\">16</span>)         │           <span style=\"color: #00af00; text-decoration-color: #00af00\">688</span> │\n",
       "├─────────────────────────────────┼────────────────────────┼───────────────┤\n",
       "│ conv1d_1 (<span style=\"color: #0087ff; text-decoration-color: #0087ff\">Conv1D</span>)               │ (<span style=\"color: #00d7ff; text-decoration-color: #00d7ff\">None</span>, <span style=\"color: #00af00; text-decoration-color: #00af00\">22</span>, <span style=\"color: #00af00; text-decoration-color: #00af00\">16</span>)         │           <span style=\"color: #00af00; text-decoration-color: #00af00\">528</span> │\n",
       "├─────────────────────────────────┼────────────────────────┼───────────────┤\n",
       "│ flatten (<span style=\"color: #0087ff; text-decoration-color: #0087ff\">Flatten</span>)               │ (<span style=\"color: #00d7ff; text-decoration-color: #00d7ff\">None</span>, <span style=\"color: #00af00; text-decoration-color: #00af00\">352</span>)            │             <span style=\"color: #00af00; text-decoration-color: #00af00\">0</span> │\n",
       "├─────────────────────────────────┼────────────────────────┼───────────────┤\n",
       "│ dense (<span style=\"color: #0087ff; text-decoration-color: #0087ff\">Dense</span>)                   │ (<span style=\"color: #00d7ff; text-decoration-color: #00d7ff\">None</span>, <span style=\"color: #00af00; text-decoration-color: #00af00\">1</span>)              │           <span style=\"color: #00af00; text-decoration-color: #00af00\">353</span> │\n",
       "└─────────────────────────────────┴────────────────────────┴───────────────┘\n",
       "</pre>\n"
      ],
      "text/plain": [
       "┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓\n",
       "┃\u001b[1m \u001b[0m\u001b[1mLayer (type)                   \u001b[0m\u001b[1m \u001b[0m┃\u001b[1m \u001b[0m\u001b[1mOutput Shape          \u001b[0m\u001b[1m \u001b[0m┃\u001b[1m \u001b[0m\u001b[1m      Param #\u001b[0m\u001b[1m \u001b[0m┃\n",
       "┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩\n",
       "│ input_layer (\u001b[38;5;33mInputLayer\u001b[0m)        │ (\u001b[38;5;45mNone\u001b[0m, \u001b[38;5;34m24\u001b[0m, \u001b[38;5;34m21\u001b[0m)         │             \u001b[38;5;34m0\u001b[0m │\n",
       "├─────────────────────────────────┼────────────────────────┼───────────────┤\n",
       "│ conv1d (\u001b[38;5;33mConv1D\u001b[0m)                 │ (\u001b[38;5;45mNone\u001b[0m, \u001b[38;5;34m23\u001b[0m, \u001b[38;5;34m16\u001b[0m)         │           \u001b[38;5;34m688\u001b[0m │\n",
       "├─────────────────────────────────┼────────────────────────┼───────────────┤\n",
       "│ conv1d_1 (\u001b[38;5;33mConv1D\u001b[0m)               │ (\u001b[38;5;45mNone\u001b[0m, \u001b[38;5;34m22\u001b[0m, \u001b[38;5;34m16\u001b[0m)         │           \u001b[38;5;34m528\u001b[0m │\n",
       "├─────────────────────────────────┼────────────────────────┼───────────────┤\n",
       "│ flatten (\u001b[38;5;33mFlatten\u001b[0m)               │ (\u001b[38;5;45mNone\u001b[0m, \u001b[38;5;34m352\u001b[0m)            │             \u001b[38;5;34m0\u001b[0m │\n",
       "├─────────────────────────────────┼────────────────────────┼───────────────┤\n",
       "│ dense (\u001b[38;5;33mDense\u001b[0m)                   │ (\u001b[38;5;45mNone\u001b[0m, \u001b[38;5;34m1\u001b[0m)              │           \u001b[38;5;34m353\u001b[0m │\n",
       "└─────────────────────────────────┴────────────────────────┴───────────────┘\n"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    },
    {
     "data": {
      "text/html": [
       "<pre style=\"white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace\"><span style=\"font-weight: bold\"> Total params: </span><span style=\"color: #00af00; text-decoration-color: #00af00\">1,569</span> (6.13 KB)\n",
       "</pre>\n"
      ],
      "text/plain": [
       "\u001b[1m Total params: \u001b[0m\u001b[38;5;34m1,569\u001b[0m (6.13 KB)\n"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    },
    {
     "data": {
      "text/html": [
       "<pre style=\"white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace\"><span style=\"font-weight: bold\"> Trainable params: </span><span style=\"color: #00af00; text-decoration-color: #00af00\">1,569</span> (6.13 KB)\n",
       "</pre>\n"
      ],
      "text/plain": [
       "\u001b[1m Trainable params: \u001b[0m\u001b[38;5;34m1,569\u001b[0m (6.13 KB)\n"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    },
    {
     "data": {
      "text/html": [
       "<pre style=\"white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace\"><span style=\"font-weight: bold\"> Non-trainable params: </span><span style=\"color: #00af00; text-decoration-color: #00af00\">0</span> (0.00 B)\n",
       "</pre>\n"
      ],
      "text/plain": [
       "\u001b[1m Non-trainable params: \u001b[0m\u001b[38;5;34m0\u001b[0m (0.00 B)\n"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "model1 = CNN()\n",
    "model1.summary()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Section 2: Model Parameters and CNN Architecture\n",
    "The following cells define input shape parameters and build the Conv1D-based CNN model for forecasting.\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 7,
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "You must install pydot (`pip install pydot`) for `plot_model` to work.\n"
     ]
    }
   ],
   "source": [
    "tensorflow.keras.utils.plot_model(model1 )"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 8,
   "metadata": {},
   "outputs": [],
   "source": [
    "checkpoints = r'D:\\Machine Learning\\ML Lab\\Lab 11\\E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5'\n",
    "OUTPUT_PATH = r'D:\\Machine Learning\\ML Lab\\Lab 11'\n",
    "FIG_PATH = os.path.sep.join([OUTPUT_PATH,\"\\history.png\"])\n",
    "JSON_PATH = os.path.sep.join([OUTPUT_PATH,\"\\history.json\"])"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 9,
   "metadata": {},
   "outputs": [],
   "source": [
    "# construct the callback to save only the *best* model to disk\n",
    "# based on the validation loss\n",
    "EpochCheckpoint1 = ModelCheckpoint(checkpoints,\n",
    "                             monitor=\"val_loss\",\n",
    "                             save_best_only=True, \n",
    "                             verbose=1)\n",
    "TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)\n",
    "\n",
    "# construct the set of callbacks\n",
    "callbacks = [EpochCheckpoint1,TrainingMonitor1]"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 10,
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "[INFO] compiling model...\n",
      "WARNING:tensorflow:TensorFlow GPU support is not available on native Windows for TensorFlow >= 2.11. Even if CUDA/cuDNN are installed, GPU will not be used. Please use WSL2 or the TensorFlow-DirectML plugin.\n"
     ]
    }
   ],
   "source": [
    "# if there is no specific model checkpoint supplied, then initialize\n",
    "# the network and compile the model\n",
    "if model is None:\n",
    "    print(\"[INFO] compiling model...\")\n",
    "    model =CNN()\n",
    "    opt = Adam(1e-3)\n",
    "    model.compile(loss= 'mae', optimizer=opt, metrics=[\"mae\", \"mape\"])\n",
    "# otherwise, load the checkpoint from disk\n",
    "else:\n",
    "    print(\"[INFO] loading {}...\".format(model))\n",
    "    model = load_model(model)\n",
    "\n",
    "    # update the learning rate\n",
    "    print(\"[INFO] old learning rate: {}\".format(K.get_value(model.optimizer.lr)))\n",
    "    K.set_value(model.optimizer.lr, 1e-4)\n",
    "    print(\"[INFO] new learning rate: {}\".format(K.get_value(model.optimizer.lr)))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 11,
   "metadata": {},
   "outputs": [
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "c:\\Users\\Yousaf Tahir\\anaconda3\\Lib\\site-packages\\sklearn\\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:\n",
      "https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations\n",
      "  warnings.warn(\n"
     ]
    },
    {
     "data": {
      "text/plain": [
       "((860, 21), (90, 21), (30, 21))"
      ]
     },
     "execution_count": 11,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "import os\n",
    "path_dataset =r'D:\\Machine Learning\\ML Lab\\Lab 11'\n",
    "path_tr = os.path.join(path_dataset, 'train.csv')\n",
    "df_tr = pd.read_csv(path_tr)\n",
    "train_set = df_tr.iloc[:].values\n",
    "path_v = os.path.join(path_dataset, 'validation.csv')\n",
    "df_v = pd.read_csv(path_v)\n",
    "validation_set = df_v.iloc[:].values \n",
    "path_te = os.path.join(path_dataset, 'test.csv')\n",
    "df_te = pd.read_csv(path_te)\n",
    "test_set = df_te.iloc[:].values \n",
    "\n",
    "path_scaler = os.path.join(path_dataset, 'AEP_scaler.pkl')\n",
    "scaler         = pickle.load(open(path_scaler, 'rb'))\n",
    "\n",
    "train_set.shape, validation_set.shape, test_set.shape"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Section 3: Training Configuration and Callback Setup\n",
    "This section prepares checkpoint saving, monitoring, optimizer configuration, and model compilation for training.\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 12,
   "metadata": {},
   "outputs": [],
   "source": [
    "time_steps=24\n",
    "num_features=21"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 13,
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Time Consumed 0.003786802291870117 sec\n"
     ]
    }
   ],
   "source": [
    "start = time.time()\n",
    "train_X , train_y = univariate_multi_step(train_set, time_steps, target_col=0,target_len=1)\n",
    "validation_X, validation_y = univariate_multi_step(validation_set, time_steps, target_col=0,target_len=1)\n",
    "test_X, test_y = univariate_multi_step(test_set, time_steps, target_col=0,target_len=1)\n",
    "print('Time Consumed', time.time()-start, \"sec\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 14,
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Epoch 1/10\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m22s\u001b[0m 865ms/step - loss: 0.2960 - mae: 0.2960 - mape: 333.7784\n",
      "Epoch 1: val_loss improved from None to 0.11112, saving model to D:\\Machine Learning\\ML Lab\\Lab 11\\E1-cp-0001-loss0.11.h5\n"
     ]
    },
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. \n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\n",
      "Epoch 1: finished saving model to D:\\Machine Learning\\ML Lab\\Lab 11\\E1-cp-0001-loss0.11.h5\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m1s\u001b[0m 11ms/step - loss: 0.1401 - mae: 0.1401 - mape: 80.6317 - val_loss: 0.1111 - val_mae: 0.1111 - val_mape: 38.9276\n",
      "Epoch 2/10\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 21ms/step - loss: 0.0917 - mae: 0.0917 - mape: 38.3603\n",
      "Epoch 2: val_loss improved from 0.11112 to 0.07348, saving model to D:\\Machine Learning\\ML Lab\\Lab 11\\E1-cp-0002-loss0.07.h5\n"
     ]
    },
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. \n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\n",
      "Epoch 2: finished saving model to D:\\Machine Learning\\ML Lab\\Lab 11\\E1-cp-0002-loss0.07.h5\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 10ms/step - loss: 0.0736 - mae: 0.0736 - mape: 37.1183 - val_loss: 0.0735 - val_mae: 0.0735 - val_mape: 24.3305\n",
      "Epoch 3/10\n",
      "\u001b[1m23/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m━━━\u001b[0m \u001b[1m0s\u001b[0m 2ms/step - loss: 0.0647 - mae: 0.0647 - mape: 35.1903 \n",
      "Epoch 3: val_loss improved from 0.07348 to 0.05543, saving model to D:\\Machine Learning\\ML Lab\\Lab 11\\E1-cp-0003-loss0.06.h5\n"
     ]
    },
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. \n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\n",
      "Epoch 3: finished saving model to D:\\Machine Learning\\ML Lab\\Lab 11\\E1-cp-0003-loss0.06.h5\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 8ms/step - loss: 0.0646 - mae: 0.0646 - mape: 33.8413 - val_loss: 0.0554 - val_mae: 0.0554 - val_mape: 17.1315\n",
      "Epoch 4/10\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 19ms/step - loss: 0.0531 - mae: 0.0531 - mape: 27.5841\n",
      "Epoch 4: val_loss did not improve from 0.05543\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0558 - mae: 0.0558 - mape: 28.4584 - val_loss: 0.0645 - val_mae: 0.0645 - val_mape: 20.1131\n",
      "Epoch 5/10\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 19ms/step - loss: 0.0524 - mae: 0.0524 - mape: 17.3207\n",
      "Epoch 5: val_loss did not improve from 0.05543\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 14ms/step - loss: 0.0544 - mae: 0.0544 - mape: 27.9865 - val_loss: 0.0607 - val_mae: 0.0607 - val_mape: 18.5301\n",
      "Epoch 6/10\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 18ms/step - loss: 0.0379 - mae: 0.0379 - mape: 15.3660\n",
      "Epoch 6: val_loss improved from 0.05543 to 0.04414, saving model to D:\\Machine Learning\\ML Lab\\Lab 11\\E1-cp-0006-loss0.04.h5\n"
     ]
    },
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. \n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\n",
      "Epoch 6: finished saving model to D:\\Machine Learning\\ML Lab\\Lab 11\\E1-cp-0006-loss0.04.h5\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0480 - mae: 0.0480 - mape: 25.0501 - val_loss: 0.0441 - val_mae: 0.0441 - val_mape: 14.1585\n",
      "Epoch 7/10\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 19ms/step - loss: 0.0516 - mae: 0.0516 - mape: 25.1835\n",
      "Epoch 7: val_loss improved from 0.04414 to 0.03583, saving model to D:\\Machine Learning\\ML Lab\\Lab 11\\E1-cp-0007-loss0.04.h5\n"
     ]
    },
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. \n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\n",
      "Epoch 7: finished saving model to D:\\Machine Learning\\ML Lab\\Lab 11\\E1-cp-0007-loss0.04.h5\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0474 - mae: 0.0474 - mape: 24.8303 - val_loss: 0.0358 - val_mae: 0.0358 - val_mape: 11.6345\n",
      "Epoch 8/10\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 19ms/step - loss: 0.0544 - mae: 0.0544 - mape: 24.5637\n",
      "Epoch 8: val_loss did not improve from 0.03583\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0428 - mae: 0.0428 - mape: 22.3335 - val_loss: 0.0438 - val_mae: 0.0438 - val_mape: 13.5205\n",
      "Epoch 9/10\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 19ms/step - loss: 0.0518 - mae: 0.0518 - mape: 22.0308\n",
      "Epoch 9: val_loss did not improve from 0.03583\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0402 - mae: 0.0402 - mape: 20.7246 - val_loss: 0.0614 - val_mae: 0.0614 - val_mape: 20.8947\n",
      "Epoch 10/10\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 18ms/step - loss: 0.0504 - mae: 0.0504 - mape: 18.6923\n",
      "Epoch 10: val_loss did not improve from 0.03583\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0396 - mae: 0.0396 - mape: 18.9241 - val_loss: 0.0425 - val_mae: 0.0425 - val_mape: 14.3556\n"
     ]
    }
   ],
   "source": [
    "epochs = 10\n",
    "verbose = 1 #0\n",
    "batch_size = 32\n",
    "History = model.fit(train_X,\n",
    "                        train_y,\n",
    "                        batch_size=batch_size,   \n",
    "                        epochs = epochs, \n",
    "                        validation_data = (validation_X,validation_y),\n",
    "                        callbacks=callbacks,verbose = verbose)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 15,
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\u001b[1m1/1\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 70ms/step\n",
      "Mean Absolute Error (MAE): 1638.59\n",
      "Median Absolute Error (MedAE): 1658.82\n",
      "Mean Squared Error (MSE): 3138565.16\n",
      "Root Mean Squared Error (RMSE): 1771.6\n",
      "Mean Absolute Percentage Error (MAPE): 10.46 %\n",
      "Median Absolute Percentage Error (MDAPE): 10.72 %\n",
      "\n",
      "\n",
      "y_test_unscaled.shape=  (5, 1)\n",
      "y_pred.shape=  (5, 1)\n"
     ]
    }
   ],
   "source": [
    "\n",
    "model = load_model(r'D:\\Machine Learning\\ML Lab\\Lab 11\\E1-cp-0007-loss0.04.h5', compile=False)\n",
    "\n",
    "y_pred_scaled   = model.predict(test_X)\n",
    "y_pred          = scaler.inverse_transform(y_pred_scaled)\n",
    "y_test_unscaled = scaler.inverse_transform(test_y)\n",
    "# Mean Absolute Error (MAE)\n",
    "MAE = np.mean(abs(y_pred - y_test_unscaled)) \n",
    "print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))\n",
    "\n",
    "# Median Absolute Error (MedAE)\n",
    "MEDAE = np.median(abs(y_pred - y_test_unscaled))\n",
    "print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))\n",
    "\n",
    "# Mean Squared Error (MSE)\n",
    "MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()\n",
    "print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))\n",
    "\n",
    "# Root Mean Squarred Error (RMSE) \n",
    "RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))\n",
    "print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))\n",
    "\n",
    "# Mean Absolute Percentage Error (MAPE)\n",
    "MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100\n",
    "print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')\n",
    "\n",
    "# Median Absolute Percentage Error (MDAPE)\n",
    "MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100\n",
    "print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')\n",
    "\n",
    "print('\\n\\ny_test_unscaled.shape= ',y_test_unscaled.shape)\n",
    "print('y_pred.shape= ',y_pred.shape)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 16,
   "metadata": {},
   "outputs": [],
   "source": [
    "checkpoints = r'D:\\Machine Learning\\ML Lab\\Lab 11\\E1-cp-0007-loss0.04.h5'\n",
    "model=r'D:\\Machine Learning\\ML Lab\\Lab 11\\E1-cp-0007-loss0.04.h5'\n",
    "start_epoch= 58"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Section 4: Dataset Loading, Prediction, and Evaluation\n",
    "The remaining cells load the data, prepare it for the CNN model, generate predictions, and calculate forecasting metrics.\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 17,
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "[INFO] loading D:\\Machine Learning\\ML Lab\\Lab 11\\E1-cp-0007-loss0.04.h5...\n",
      "[INFO] old learning rate: 9.999999747378752e-05\n",
      "[INFO] new learning rate: 9.999999747378752e-05\n"
     ]
    }
   ],
   "source": [
    "# construct the callback to save only the *best* model to disk\n",
    "# based on the validation loss\n",
    "EpochCheckpoint1 = ModelCheckpoint(checkpoints,\n",
    "                             monitor=\"val_loss\",\n",
    "                             save_best_only=True, \n",
    "                             verbose=1)\n",
    "TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)\n",
    "\n",
    "# construct the set of callbacks\n",
    "callbacks = [EpochCheckpoint1,TrainingMonitor1]\n",
    "model_path = r'D:\\Machine Learning\\ML Lab\\Lab 11\\E1-cp-0007-loss0.04.h5'\n",
    "# if there is no specific model checkpoint supplied, then initialize\n",
    "# the network and compile the model\n",
    "if model is None:\n",
    "    print(\"[INFO] compiling model...\")\n",
    "    model = PC.build(time_steps=24, num_features=21, reg=0.0005)\n",
    "    opt = Adam(1e-3)\n",
    "    model.compile(loss= 'mae', optimizer=opt, metrics=[\"mae\", \"mape\"])\n",
    "# otherwise, load the checkpoint from disk\n",
    "else:\n",
    "    print(\"[INFO] loading {}...\".format(model_path))\n",
    "    model = load_model(model_path, compile=False)\n",
    "\n",
    "    opt = Adam(learning_rate=1e-4)\n",
    "    model.compile(\n",
    "        loss=\"mae\",\n",
    "        optimizer=opt,\n",
    "        metrics=[\"mae\", \"mape\"]\n",
    "    )\n",
    "\n",
    "    # update the learning rate\n",
    "    print(\"[INFO] old learning rate: {}\".format(K.get_value(model.optimizer.learning_rate)))\n",
    "    print(\"[INFO] new learning rate: {}\".format(K.get_value(model.optimizer.learning_rate)))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 18,
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Epoch 1/10\n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m18s\u001b[0m 710ms/step - loss: 0.0413 - mae: 0.0413 - mape: 41.4780\n",
      "Epoch 1: val_loss improved from None to 0.04066, saving model to D:\\Machine Learning\\ML Lab\\Lab 11\\E1-cp-0007-loss0.04.h5\n"
     ]
    },
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. \n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\n",
      "Epoch 1: finished saving model to D:\\Machine Learning\\ML Lab\\Lab 11\\E1-cp-0007-loss0.04.h5\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m1s\u001b[0m 12ms/step - loss: 0.0409 - mae: 0.0409 - mape: 20.8832 - val_loss: 0.0407 - val_mae: 0.0407 - val_mape: 13.5609\n",
      "Epoch 2/10\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 18ms/step - loss: 0.0375 - mae: 0.0375 - mape: 20.0464\n",
      "Epoch 2: val_loss improved from 0.04066 to 0.03947, saving model to D:\\Machine Learning\\ML Lab\\Lab 11\\E1-cp-0007-loss0.04.h5\n"
     ]
    },
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. \n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\n",
      "Epoch 2: finished saving model to D:\\Machine Learning\\ML Lab\\Lab 11\\E1-cp-0007-loss0.04.h5\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0398 - mae: 0.0398 - mape: 20.1133 - val_loss: 0.0395 - val_mae: 0.0395 - val_mape: 13.0236\n",
      "Epoch 3/10\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 23ms/step - loss: 0.0491 - mae: 0.0491 - mape: 29.7032\n",
      "Epoch 3: val_loss did not improve from 0.03947\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0393 - mae: 0.0393 - mape: 19.8673 - val_loss: 0.0407 - val_mae: 0.0407 - val_mape: 13.5255\n",
      "Epoch 4/10\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 18ms/step - loss: 0.0386 - mae: 0.0386 - mape: 15.6954\n",
      "Epoch 4: val_loss did not improve from 0.03947\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0388 - mae: 0.0388 - mape: 19.4345 - val_loss: 0.0443 - val_mae: 0.0443 - val_mape: 14.8280\n",
      "Epoch 5/10\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 19ms/step - loss: 0.0392 - mae: 0.0392 - mape: 16.1246\n",
      "Epoch 5: val_loss did not improve from 0.03947\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0383 - mae: 0.0383 - mape: 20.1498 - val_loss: 0.0426 - val_mae: 0.0426 - val_mape: 14.2441\n",
      "Epoch 6/10\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 18ms/step - loss: 0.0342 - mae: 0.0342 - mape: 12.6033\n",
      "Epoch 6: val_loss did not improve from 0.03947\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0383 - mae: 0.0383 - mape: 20.2231 - val_loss: 0.0414 - val_mae: 0.0414 - val_mape: 13.8553\n",
      "Epoch 7/10\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 20ms/step - loss: 0.0390 - mae: 0.0390 - mape: 23.9750\n",
      "Epoch 7: val_loss did not improve from 0.03947\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0378 - mae: 0.0378 - mape: 19.5724 - val_loss: 0.0396 - val_mae: 0.0396 - val_mape: 13.1926\n",
      "Epoch 8/10\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 20ms/step - loss: 0.0396 - mae: 0.0396 - mape: 21.5370\n",
      "Epoch 8: val_loss did not improve from 0.03947\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0374 - mae: 0.0374 - mape: 19.7232 - val_loss: 0.0423 - val_mae: 0.0423 - val_mape: 14.1391\n",
      "Epoch 9/10\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 19ms/step - loss: 0.0289 - mae: 0.0289 - mape: 23.3246\n",
      "Epoch 9: val_loss did not improve from 0.03947\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0369 - mae: 0.0369 - mape: 19.3079 - val_loss: 0.0409 - val_mae: 0.0409 - val_mape: 13.6620\n",
      "Epoch 10/10\n",
      "\u001b[1m 1/27\u001b[0m \u001b[37m━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[1m0s\u001b[0m 20ms/step - loss: 0.0412 - mae: 0.0412 - mape: 17.5589\n",
      "Epoch 10: val_loss did not improve from 0.03947\n",
      "\u001b[1m27/27\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0362 - mae: 0.0362 - mape: 19.2013 - val_loss: 0.0421 - val_mae: 0.0421 - val_mape: 14.0183\n"
     ]
    }
   ],
   "source": [
    "epochs = 10\n",
    "verbose = 1 #0\n",
    "batch_size = 32\n",
    "History = model.fit(train_X,\n",
    "                        train_y,\n",
    "                        batch_size=batch_size,   \n",
    "                        epochs = epochs, \n",
    "                        validation_data = (validation_X,validation_y),\n",
    "                        callbacks=callbacks,\n",
    "                        verbose = verbose)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 19,
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\u001b[1m1/1\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 54ms/step\n",
      "Mean Absolute Error (MAE): 1504.02\n",
      "Median Absolute Error (MedAE): 1526.72\n",
      "Mean Squared Error (MSE): 2685976.07\n",
      "Root Mean Squared Error (RMSE): 1638.89\n",
      "Mean Absolute Percentage Error (MAPE): 9.6 %\n",
      "Median Absolute Percentage Error (MDAPE): 9.87 %\n",
      "\n",
      "\n",
      "y_test_unscaled.shape=  (5, 1)\n",
      "y_pred.shape=  (5, 1)\n"
     ]
    }
   ],
   "source": [
    "\n",
    "model = load_model(r'D:\\Machine Learning\\ML Lab\\Lab 11\\E1-cp-0007-loss0.04.h5', compile=False)\n",
    "\n",
    "y_pred_scaled   = model.predict(test_X)\n",
    "y_pred          = scaler.inverse_transform(y_pred_scaled)\n",
    "y_test_unscaled = scaler.inverse_transform(test_y)\n",
    "# Mean Absolute Error (MAE)\n",
    "MAE = np.mean(abs(y_pred - y_test_unscaled)) \n",
    "print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))\n",
    "\n",
    "# Median Absolute Error (MedAE)\n",
    "MEDAE = np.median(abs(y_pred - y_test_unscaled))\n",
    "print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))\n",
    "\n",
    "# Mean Squared Error (MSE)\n",
    "MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()\n",
    "print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))\n",
    "\n",
    "# Root Mean Squarred Error (RMSE) \n",
    "RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))\n",
    "print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))\n",
    "\n",
    "# Mean Absolute Percentage Error (MAPE)\n",
    "MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100\n",
    "print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')\n",
    "\n",
    "# Median Absolute Percentage Error (MDAPE)\n",
    "MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100\n",
    "print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')\n",
    "\n",
    "print('\\n\\ny_test_unscaled.shape= ',y_test_unscaled.shape)\n",
    "print('y_pred.shape= ',y_pred.shape)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Final Conclusion\n",
    "In this lab, a CNN model was applied to time-series forecasting. The notebook shows how Conv1D layers can extract local temporal patterns and how the trained model is evaluated using regression metrics."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": []
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "base",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.13.9"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}